# Run SurEau

In [ ]:
# | default_exp run_sureau

In [ ]:
# | hide
from fastcore import *
from nbdev.showdoc import *

In [ ]:
# | export
import copy
import numpy as np
import pandas as pd
from dateutil import parser
from plant_hydraulics.parameter_classes import (
    SurEauVegetationParams,
    SurEauSoilParams,
    SurEauModelOptions,
)

from plant_hydraulics.sureau_soil_params import (
    sureau_soil_params,
    compute_soil_root_geometry,
)

from plant_hydraulics.sureau_object_initialization import (
    init_soil,
    init_plant,
    yearly_init_plant,
)
from plant_hydraulics.sureau_climate import (
    new_climate_day,
    compute_Rn_and_ETP,
    new_climate_hourly,
    get_hourly_snapshot,
    interpolate_climate_hourly,
)
from plant_hydraulics.sureau_vegetation_processes import (
    compute_pheno,
    update_LAI_and_stocks,
    compute_interception,
    compute_evapo_intercepted,
    compute_transpiration,
    compute_water_storage,
    update_kplant,
    update_capacitances,
   
)
from plant_hydraulics.sureau_soil_hydraulics import (
    compute_infiltration,
    compute_evaporation,
    update_soil_water,
    set_SWC_to_field_capacity,
)
from plant_hydraulics.utils import flux_leaf_to_stand
from plant_hydraulics.sureau_solver import sureau_solver
from plant_hydraulics.sureau_plant_hydraulics import compute_regul_fact
from plant_hydraulics.sureau_vegetation_params import sureau_vegetation_params

In [ ]:
# | export


def _collect_timestep(state, fluxes, diag, soil, clim, year, doy):
    """Collect key variables into a dict for DataFrame construction.
 
    Pulls from the correct object for each variable:
        state  → water potentials, PLC, conductances, LAI
        fluxes → transpiration rates, stomatal conductance, leaf temperature
        diag   → fuel moisture content, solver quality metrics
    """
    return {
        "year": year,
        "doy": doy,
        "time": clim.get("time", np.nan),
        "T_air": clim.get("T_air_mean", np.nan),
        "VPD": clim.get("VPD", np.nan),
        "PAR": clim.get("PAR", np.nan),
        
        # From STATE (persistent, carried forward) ------------------------------
        "psi_LApo": state.psi_LApo,
        "psi_SApo": state.psi_SApo,
        "psi_LSym": state.psi_LSym,
        "psi_SSym": state.psi_SSym,
        "psi_soil_mean": state.psi_all_soil,
        "PLC_leaf": state.PLC_leaf,
        "PLC_stem": state.PLC_stem,
        "LAI": state.LAI,
        "k_plant": state.k_plant,
        
        # From FLUXES (instantaneous, recomputed each step) ---------------------
        "gs_lim": fluxes.gs_lim,
        "E_lim": fluxes.E_lim,
        "E_min": fluxes.E_min,
        "E_bound": fluxes.E_bound,
        "transpiration_mm": fluxes.transpiration_mm,
        "regul_fact": fluxes.regul_fact,
        "leaf_temperature": fluxes.leaf_temperature,
        
        # From DIAGNOSTICS (logged, not fed back) -------------------------------
        "LFMC": diag.LFMC,
        "diag_timestep_h": diag.diag_timestep_hours,
        
        # From SOIL -------------------------------------------------------------
        "soil_water_stock_total": np.sum(soil.soil_water_stock),
        "REW_mean": np.mean(soil.REW),
        "soil_evaporation": soil.evaporation,
        "drainage": soil.drainage,
    }
 

In [ ]:
# | export

def run_sureau(
    climate_df: pd.DataFrame,
    veg_params: SurEauVegetationParams,
    soil_params: SurEauSoilParams,
    opts: SurEauModelOptions,
    deep_water: bool = False,
) -> pd.DataFrame:
    """
    Run the SurEau-Ecos plant hydraulics simulation.
 
    This is the **top-level simulation driver** that orchestrates the
    entire SurEau-Ecos model (Ruffault et al. 2022). It loops over
    years, days, and hours, calling every component function in the
    correct order to simulate the soil-plant-atmosphere water
    continuum, predict drought-induced hydraulic failure, and compute
    fuel moisture content for wildfire risk assessment.
 
    The simulation has three nested time scales:
 
    **Yearly**: Reset soil water (optional), restart phenological
    cycle (budburst accumulator, LAI).
 
    **Daily**: Update phenology, LAI and water storage volumes, capacitances,
    radiation and ETP, rainfall interception, soil infiltration, and plant 
    conductances.
 
    **Hourly** (with adaptive sub-stepping): Partition ETP between canopy and 
    soil, evaporate intercepted water, then solve the four-compartment hydraulic 
    ODE system using the selected numerical scheme. An adaptive sub-stepping 
    loop refines the timestep until stomatal regulation (γ) changes by 
    < 5% and PLC changes by < 1% per sub-step.
 
    The simulation ends when the climate data is exhausted or when
    leaf PLC exceeds the mortality threshold (default 90%).
 
    __Parameters:__
 
    - climate_df: pandas DataFrame with daily climate forcing. Must
        contain columns:
 
        - Year: Calendar year (int).
        - Doy: Day of year (int, 1–365).
        - DATE: Date string or datetime.
        - Tair_mean, Tair_max, Tair_min: Air temperature (°C).
        - RHair_mean, RHair_max, RHair_min: Relative humidity (%).
        - PPT_sum: Daily precipitation (mm).
        - RG_sum: Daily global radiation (MJ/m²/day or W/m²,
            depending on ``opts``).
        - WS_mean: Daily mean wind speed (m/s).
 
    - veg_params: SurEauVegetationParams object. Species-specific
        parameters including: LAI_max, LMA, LDMC (leaf traits);
        P50_VC_leaf/stem, slope_VC_leaf/stem (vulnerability curves);
        P12_gs, P88_gs (stomatal regulation); gmin20, TPhase_gmin,
        Q10_1/2_gmin (cuticular conductance); beta_root_profile or
        root_Z50/Z95 (root distribution); and many others.
        Derived parameters (root distribution, conductance
        partitioning, etc.) are computed internally by
        ``sureau_vegetation_params``.
 
    - soil_params: SurEauSoilParams object. Soil parameters including:
        depth (cumulative layer depths), texture parameters (alpha, n
        for van Genuchten), theta_sat, theta_res (porosity, residual
        water), RFC (rock fragment content), and initial water content.
        Derived parameters (layer thickness, field capacity, etc.)
        are computed internally by ``sureau_soil_params``.
 
    - opts: SurEauModelOptions object controlling the simulation:
 
        - year_start, year_end: Simulation period.
        - print_progress: Whether to print progress to console.
        - comp_options: SurEauComputationOptions with:
            - numerical_scheme: ``"Implicit"`` (default),
                ``"Semi-Implicit"``, or ``"Explicit"``.
            - n_small_timesteps: List of sub-timestep counts for
                adaptive stepping, e.g. ``[1, 2, 4, 8]``.
            - Lsym, Ssym, CLapo, CTapo: Capacitance scaling flags.
            - Lcav, Scav: Cavitation flux on/off.
            - Eord: Transpiration linearization on/off (Eq. 60).
 
    - deep_water: If True, keep the deepest soil layer permanently
        at field capacity, simulating access to a groundwater table.
        Default False. Useful for isolating atmospheric drought
        (high VPD) from soil moisture limitation.
 
    __Returns:__
 
    - pd.DataFrame: Simulation results at hourly (or sub-hourly)
        resolution. Each row is one timestep. Columns include:
 
        Water potentials: psi_LApo, psi_SApo, psi_LSym, psi_SSym,
        psi_all_soil (MPa).
 
        Cavitation: PLC_leaf, PLC_stem (%).
 
        Transpiration: E_lim, E_min, E_min_S (mmol/m²/s);
        transpiration_mm (mm/timestep).
 
        Stomatal regulation: regul_fact (γ, 0–1), gs_lim (mmol/m²/s).
 
        Fuel moisture: LFMC, DFMC, FMC_canopy (%).
 
        Soil: SWC per layer, psi_soil per layer.
 
        Diagnostics: leaf_temperature, diag_delta_regul_max,
        diag_delta_PLC_max, diag_timestep_hours.
 
 
    __Simulation structure:__
 
        ┌─────────────────────────────────────────────────────────┐
        │  INITIALIZATION (once)                                  │
        │                                                         │
        │  sureau_soil_params     → derived soil properties       │
        │  sureau_vegetation_params → root profile (Eq. 19),      │
        │                            root length (Eq. 18),        │
        │                            conductance split (Eq. 17)   │
        │  compute_soil_root_geometry → Gardner-Cowan B_GC (Eq.21)│
        │  init_plant, init_soil  → initial state objects         │
        └───────────────────────────┬─────────────────────────────┘
                                    │
                                    ▼
        ┌─────────────────────────────────────────────────────────┐
        │  YEARLY LOOP                                            │
        │                                                         │
        │  Optional: reset soil to field capacity                 │
        │  yearly_init_plant → reset phenology accumulators       │
        │                                                         │
        │  ┌──────────────────────────────────────────────────┐   │
        │  │  DAILY LOOP                                      │   │
        │  │                                                  │   │
        │  │  new_climate_day       → daily climate snapshot  │   │
        │  │  compute_pheno         → LAI_pheno (Eqs. A1–A2)  │   │
        │  │  update_LAI_and_stocks → LAI, Q_sat (Eqs. 36–40) │   │
        │  │  compute_Rn_and_ETP   → net radiation, ETP       │   │
        │  │  compute_interception → ppt_soil, intercepted    │   │
        │  │  compute_infiltration → soil water update        │   │
        │  │  update_kplant        → conductances (Eqs. 13–20)│   │
        │  │                                                  │   │
        │  │  ┌───────────────────────────────────────────┐   │   │
        │  │  │  HOURLY LOOP                              │   │   │
        │  │  │                                           │   │   │
        │  │  │  ETP partitioning (canopy vs soil)        │   │   │
        │  │  │  compute_evapo_intercepted → ETP_r        │   │   │
        │  │  │                                           │   │   │
        │  │  │  ┌────────────────────────────────────┐   │   │   │
        │  │  │  │  ADAPTIVE SUB-STEPPING             │   │   │   │
        │  │  │  │                                    │   │   │   │
        │  │  │  │  while Δγ > 5% or ΔPLC > 1%:       │   │   │   │
        │  │  │  │    deep copy state                 │   │   │   │
        │  │  │  │    for each sub-timestep:          │   │   │   │
        │  │  │  │      compute_transpiration         │   │   │   │
        │  │  │  │                                    │   │   │   │
        │  │  │  │      sureau_solver ← THE CORE      │   │   │   │
        │  │  │  │                                    │   │   │   │
        │  │  │  │      update_kplant                 │   │   │   │
        │  │  │  │      update_capacitances           │   │   │   │
        │  │  │  │      update_soil_water             │   │   │   │
        │  │  │  │    check Δγ, ΔPLC                  │   │   │   │
        │  │  │  │    if ok → accept; else refine     │   │   │   │
        │  │  │  └────────────────────────────────────┘   │   │   │
        │  │  │                                           │   │   │
        │  │  │  compute_transpiration                    │   │   │
        │  │  │  flux conversions (mmol → mm)             │   │   │
        │  │  │  compute_water_storage → LFMC, FMC_canopy │   │   │
        │  │  │  mortality check (PLC > threshold?)       │   │   │
        │  │  │  collect results → append to list         │   │   │
        │  │  │                                           │   │   │
        │  │  └───────────────────────────────────────────┘   │   │
        │  │                                                  │   │
        │  └──────────────────────────────────────────────────┘   │
        │                                                         │
        └─────────────────────────────────────────────────────────┘
                                    │
                                    ▼
                          pd.DataFrame(results)
 
    __Adaptive sub-stepping strategy:__
 
    The solver tries progressively finer sub-timesteps until the
    solution quality is acceptable:
 
        ts_list = [1, 2, 4, 8]   (example)
 
        Attempt 1: nts=1 → full hour as one timestep
            Solve → check Δγ and ΔPLC
            If Δγ < 5% AND ΔPLC < 1% → accept
            Else → discard, retry with finer resolution
 
        Attempt 2: nts=2 → two 30-min sub-steps
            Deep copy original state → re-solve from scratch
            Solve → check → accept or refine further
 
        Attempt 3: nts=4 → four 15-min sub-steps
            ... and so on until ts_list is exhausted
 
    Each attempt starts from a fresh ``copy.deepcopy`` of the
    original state, ensuring failed attempts leave no trace. The
    quality criteria are:
 
    - Δγ < 0.05: Stomatal regulation factor must not change by
      more than 5% in a single sub-step. This ensures the steep
      sigmoid transition zone (Eq. 34) is resolved accurately.
 
    - ΔPLC < 1.0: Percent loss of conductivity must not change
      by more than 1% in a single sub-step. This ensures the
      cavitation vulnerability curve (Eq. 15) is resolved.
 
    __Soil water balance:__
 
    The soil is always solved explicitly.
    The soil-to-stem flux is computed as:
 
        flux = k_soil_to_stem × (ψ_soil − ψ_SApo)
 
    then converted from per-leaf-area to per-ground-area units via
    `flux_leaf_to_stand` (multiply by LAI, convert mmol → mm),
    and subtracted from the soil water content in
    `update_soil_water`.
 
    __Mortality criterion:__
 
    The simulation checks each hourly timestep whether
    ``PLC_leaf >= threshold_mortality`` (default 90%). If so, the
    tree is declared dead, the daily and hourly loops break, and
    the simulation advances to the next year (or ends). This is the
    hydraulic failure criterion — when 90% of the leaf xylem is
    embolized, the remaining 10% cannot supply enough water to
    sustain any physiological function.
 
    __Deep water option:__
 
    When ``deep_water=True``, the deepest soil layer is refilled
    to field capacity both at the start of each day and at the end
    of each hourly timestep. This simulates a shallow water table
    that the roots can always access, isolating atmospheric drought
    effects (high VPD, high temperature) from soil drying effects.
    Useful for sensitivity analysis: "would this tree survive if
    it had unlimited deep water?"
 
    __Key outputs for applications:__
 

        Drought mortality prediction:
            PLC_leaf, PLC_stem   → hydraulic damage (irreversible)
            psi_LSym, psi_LApo   → water stress severity
            regul_fact (γ)       → stomatal closure degree
 
        Wildfire risk assessment:
            LFMC                 → live fuel moisture (< 80% = danger)
            DFMC                 → dead fuel moisture
            FMC_canopy           → canopy fuel moisture (live + dead)
 
        Water balance:
            transpiration_mm     → total plant water loss
            flux_soil_to_stem_mm → soil water uptake per layer
            soil.SWC             → soil water content per layer
 
    Parameters
    ----------
    climate_df : pd.DataFrame
        Daily climate forcing. Required columns: Year, Doy, DATE,
        Tair_mean, Tair_max, Tair_min, RHair_mean, RHair_max,
        RHair_min, PPT_sum, RG_sum, WS_mean.
    veg_params : SurEauVegetationParams
        Species-specific vegetation parameters. Derived parameters
        are computed internally.
    soil_params : SurEauSoilParams
        Soil parameters. Derived parameters are computed internally.
    opts : SurEauModelOptions
        Simulation control options (years, numerical scheme,
        adaptive stepping, output resolution).
    deep_water : bool
        If True, keep deepest soil layer at field capacity.
        Default False.
 
    Returns
    -------
    pd.DataFrame
        Simulation results at hourly resolution. One row per
        timestep, with columns for all state variables, fluxes,
        and diagnostics.
    """
    
    # Initialise derived parameters ---------------------------------------------
    # Ensure soil derived quantities exist (layer_thickness, m, 
    # V_field_capacity, etc.)
    if soil_params.layer_thickness is None:
        soil_params = sureau_soil_params(soil_params)
 
    veg_params = sureau_vegetation_params(veg_params, soil_params)
    soil_params = compute_soil_root_geometry(soil_params, veg_params)

    # Initialise state objects --------------------------------------------------
    state, fluxes, diag = init_plant(veg_params)
    
    soil = init_soil(soil_params)

    # Result collection ---------------------------------------------------------
    results = []

    # Main 
    # Year loop -----------------------------------------------------------------
    parsed_dates = climate_df["DATE"].apply(lambda d: parser.parse(d, dayfirst=True))
    climate_df = climate_df.copy()
    climate_df["_year"] = parsed_dates.apply(lambda d: d.year)
    climate_df["_doy"] = parsed_dates.apply(lambda d: d.timetuple().tm_yday)
    
    for each_YEAR in range(opts.year_start, opts.year_end + 1):
        
        # Flag for tree mortality
        stop_dead = False

        # Reset soil water content to field capacity at the start of each year 
        if soil_params.reset_SWC:
            soil = set_SWC_to_field_capacity(soil, soil_params)

        # Reset year-dependent plant state: reset the phenological cycle 
        state, fluxes, diag = yearly_init_plant(state, fluxes, diag, veg_params)

        # Get all dates for this year from the climate DataFrame
        year_mask = climate_df["_year"] == each_YEAR
        year_dates = climate_df.loc[year_mask, "DATE"].values
        year_doys = climate_df.loc[year_mask, "_doy"].values
        
        # Daily loop ------------------------------------------------------------
        for day_idx, date_str in enumerate(year_dates):
            
            each_DAY = year_doys[day_idx]
        
            if opts.print_progress:
                
                # Progress indicator 
                print(f"\rYear {each_YEAR} Day {each_DAY:3d}", end="", flush=True)

            # Daily climate
            clim_day = new_climate_day(climate_df, date_str)

            # Phenology and LAI
            state = compute_pheno(state, veg_params, clim_day.T_air_mean, each_DAY)
            state = update_LAI_and_stocks(state, veg_params)

            # Net radiation and ETP
            clim_day = compute_Rn_and_ETP(clim_day, veg_params, opts)

            # Interception
            fluxes = compute_interception(state, fluxes, clim_day.PPT)

            # Deep water option
            if deep_water:
                soil = set_SWC_to_field_capacity(
                    soil, soil_params, layers=np.array([soil_params.n_layers - 1])
                )

            # Infiltration
            soil = compute_infiltration(soil, soil_params, fluxes.ppt_soil)

            # Hourly climate disaggregation
            clim_hour = new_climate_hourly(clim_day, opts, veg_params)

            # Update k_plant
            state = update_kplant(state, soil, veg_params)

            # Hourly loop -------------------------------------------------------
            for each_tt in range(len(clim_hour.ETP)):
                
                # Climate interpolation
                idx_curr = max(each_tt - 1, 0)
                idx_next = each_tt
                snap_curr = get_hourly_snapshot(clim_hour, idx_curr)
                snap_next = get_hourly_snapshot(clim_hour, idx_next)
                snap_mid = interpolate_climate_hourly(snap_curr, snap_next, 0.5)

                N_hours = clim_hour.n_hours[each_tt]

                # Split ETP between vegetation and soil
                K_lai = veg_params.K
                frac_veg = 1 - np.exp(-K_lai * state.LAI)
                fluxes.ETP = frac_veg * snap_mid["ETP"]
                soil.ETP = (1 - frac_veg) * snap_mid["ETP"]

                # Evaporation of intercepted water
                fluxes = compute_evapo_intercepted(fluxes)

                if fluxes.ETP_r > 0:
                    snap_curr["ETP_veg"] = frac_veg * snap_curr["ETP"]
                    snap_next["ETP_veg"] = frac_veg * snap_next["ETP"]
                else:
                    snap_curr["ETP_veg"] = 0.0
                    snap_next["ETP_veg"] = 0.0

                # Solver with adaptive sub-stepping 
                reg_ok = False
                cav_ok = False
                nw_comp = 0
                ts_list = opts.comp_options.n_small_timesteps

                while (not reg_ok or not cav_ok) and nw_comp < len(ts_list):
                    
                    # Copy objects
                    state_iter = copy.deepcopy(state)
                    fluxes_iter = copy.deepcopy(fluxes)
                    diag_iter = copy.deepcopy(diag)
                    soil_iter = copy.deepcopy(soil)
 
                    reg_ok = cav_ok = False
                    nw_comp += 1
                    delta_regul_max = 1e-100
                    delta_PLC_max = 1e-100
                    nts = ts_list[nw_comp - 1]
                    flux_soil_layers_large = np.zeros(soil_params.n_layers)

                    # Inner sub-timestep loop -----------------------------------
                    for its in range(nts):
                        p = (its + 0.5) / nts
                        snap = interpolate_climate_hourly(snap_curr, snap_next, p)
                        snap["ETP_veg"] = snap.get("ETP_veg", snap["ETP"] * frac_veg)

                        # Soil evaporation on small timestep
                        if soil_params.soil_evap:
                            soil_iter = compute_evaporation(
                                soil_iter,
                                soil_params,
                                snap["T_air_mean"],
                                snap["RH_air_mean"],
                                N_hours / nts,
                            )

                        # Transpiration
                        state_tmp = copy.deepcopy(state_iter)
                        fluxes_tmp = compute_transpiration(
                            state_tmp,
                            copy.deepcopy(fluxes_iter),
                            veg_params, snap, N_hours
                        )

                        # Solver ------------------------------------------------
                        # Implicit integration
                        state_np1 = sureau_solver(
                            state_iter, fluxes_tmp, diag_iter,
                            soil_iter, veg_params,
                            N_hours * 3600 / nts,
                            opts.comp_options,
                        )
                        
                        state_np1 = update_kplant(state_np1, soil_iter, veg_params)
                        state_np1 = update_capacitances(state_np1, veg_params)

                        # Check resolution quality
                        rf_np1, _ = compute_regul_fact(state_np1.psi_LSym, veg_params)
                        rf_n, _ = compute_regul_fact(state_iter.psi_LSym, veg_params)
                        delta_regul_max = max(delta_regul_max, abs(rf_np1 - rf_n))
                        
                        delta_PLC_max = max(
                            delta_PLC_max,
                            state_np1.PLC_leaf - state_iter.PLC_leaf,
                            state_np1.PLC_stem - state_iter.PLC_stem,
                        )
 

                        # Soil-to-stem flux
                        flux_sts = state_iter.k_soil_to_stem * (
                            soil_iter.psi_soil - state_np1.psi_SApo
                        )
                        fluxes_tmp.flux_soil_to_stem = flux_leaf_to_stand(
                            flux_sts, LAI=state.LAI, dt=N_hours / nts
                        )
                        soil_iter = update_soil_water(
                            soil_iter, soil_params, fluxes_tmp.flux_soil_to_stem
                        )
                        flux_soil_layers_large += flux_sts / nts
 
                        # Advance iteration state
                        state_iter = state_np1

                    # Check
                    diag_iter.diag_delta_regul_max = delta_regul_max
                    diag_iter.diag_delta_PLC_max = delta_PLC_max
                    diag_iter.diag_timestep_hours = N_hours / nts
                    reg_ok = delta_regul_max < 0.05
                    cav_ok = delta_PLC_max < 1.0

                # Finalise timestep 
                state = state_iter
                diag = diag_iter
                soil = soil_iter
                
                # Recompute transpiration at the final accepted state
                # so that fluxes (gs_lim, regul_fact, E_lim, T_leaf, 
                # etc.) reflect the converged solution before recording.
                fluxes = compute_transpiration(
                    state, fluxes, veg_params, snap_next, N_hours
                )

                # Final transpiration update
                fluxes.E_min_mm = flux_leaf_to_stand(
                    fluxes.E_min, LAI=state.LAI, dt=N_hours
                )
                fluxes.E_min_S_mm = flux_leaf_to_stand(
                    fluxes.E_min_S, LAI=state.LAI, dt=N_hours
                )
                fluxes.sum_flux_soil_to_stem = np.sum(flux_soil_layers_large)
                fluxes.flux_soil_to_stem_mm = flux_leaf_to_stand(
                    flux_soil_layers_large, LAI=state.LAI, dt=N_hours
                )
                fluxes.transpiration_mm = flux_leaf_to_stand(
                    fluxes.E_min + fluxes.E_min_S + fluxes.E_lim,
                    LAI=state.LAI, dt=N_hours,
                )

                # Water storage
                state, diag = compute_water_storage(
                    state, diag, veg_params, snap_next["VPD"]
                )

                # Deep water
                if deep_water:
                    soil = set_SWC_to_field_capacity(
                        soil, soil_params,
                        layers=np.array([soil_params.n_layers - 1]),
                    )

                # Collect results -----------------------------------------------
                results.append(_collect_timestep(
                    state, fluxes, diag, soil, snap_next, each_YEAR, each_DAY
                ))

                # Check mortality
                if state.PLC_leaf >= veg_params.threshold_mortality:
                    stop_dead = True
                    break
 

            if stop_dead:
                if opts.print_progress:
                    print(f"\rYear {each_YEAR} Day {each_DAY}: plant is dead.")
                break

        if opts.print_progress:
            print(f"\rYear {each_YEAR} complete. ")

    return pd.DataFrame(results)